# Notes


LunarLander:

States: 8 continuous values
Actions: 4 discrete
Solved: average reward $\geq$ 200 over 100 episodes

# -

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np

from env_utils import make_env, set_seed, get_device, RunLogger

In [12]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        logits = self.network(x)
        return Categorical(logits=logits) # Distribution

In [13]:
def select_action(policy, state, device):
    state_tensor = torch.FloatTensor(state).to(device)
    dist = policy(state_tensor)
    action = dist.sample()              # Sample from distribution 
    log_prob = dist.log_prob(action)    # Saving for loss 
    return action.item(), log_prob

In [14]:
def compute_returns(rewards, gamma=0.99):
    returns = []
    G = 0
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    returns = torch.FloatTensor(returns)
    # Normalized returns
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns

In [21]:
def update_policy(optimizer, log_probs, returns):
    loss = torch.stack([-log_prob * G for log_prob, G in zip(log_probs, returns)]).sum()    # Negative reward because we maximize loss 

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item()

In [22]:
def train_reinforce(seed=42, num_episodes=3000, gamma=0.99, lr=3e-4):

    set_seed(seed)
    device = get_device()
    env, env_info = make_env(seed=seed, continuous=False)

    print(f"Env: {env_info.env_name}")
    print(f"State dim: {env_info.obs_dim}, Actions: {env_info.n_actions}")
    print(f"Device: {device}")

    policy = PolicyNetwork(
        state_dim=env_info.obs_dim,
        action_dim=env_info.n_actions,
        hidden_dim=128
    ).to(device)

    optimizer = optim.Adam(policy.parameters(), lr=lr)

    with RunLogger("reinforce", seed=seed, enable_tb=False) as logger:
        rewards_per_episode = []

        for episode in range(num_episodes):
            state, _ = env.reset()
            log_probs = []
            rewards = []
            done = False
            steps = 0

            while not done:
                action, log_prob = select_action(policy, state, device)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated

                log_probs.append(log_prob)
                rewards.append(reward)
                state = next_state
                steps += 1

            returns = compute_returns(rewards, gamma)
            loss = update_policy(optimizer, log_probs, returns)

            total_reward = sum(rewards)
            rewards_per_episode.append(total_reward)

            logger.log_episode(episode=episode, reward=total_reward, length=steps, loss=loss)

            if (episode + 1) % 100 == 0:
                avg = np.mean(rewards_per_episode[-100:])
                print(f"Episode {episode+1} | Avg Reward (last 100): {avg:.2f} | Loss: {loss:.4f}")

    env.close()
    return policy, rewards_per_episode

Environment fully solved at $\sim$ 2600-3000 episodes 

In [23]:
policy, rewards = train_reinforce(seed=42, num_episodes=3000)

# Save outputs for Person 5
np.save("reinforce_rewards.npy", np.array(rewards))
torch.save(policy.state_dict(), "reinforce_policy.pth")
print("Saved: reinforce_rewards.npy, reinforce_policy.pth")
print("CSV logs saved to: results/logs/reinforce_seed42.csv")

Env: LunarLander-v3
State dim: 8, Actions: 4
Device: cpu
Episode 100 | Avg Reward (last 100): -161.35 | Loss: -0.3340
Episode 200 | Avg Reward (last 100): -147.41 | Loss: -2.7251
Episode 300 | Avg Reward (last 100): -161.39 | Loss: -11.7109
Episode 400 | Avg Reward (last 100): -148.78 | Loss: 0.6102
Episode 500 | Avg Reward (last 100): -82.43 | Loss: -42.8163
Episode 600 | Avg Reward (last 100): -78.02 | Loss: -1.8213
Episode 700 | Avg Reward (last 100): -44.62 | Loss: 5.5275
Episode 800 | Avg Reward (last 100): -85.51 | Loss: 17.1666
Episode 900 | Avg Reward (last 100): -45.36 | Loss: -5.8743
Episode 1000 | Avg Reward (last 100): -17.12 | Loss: -5.5606
Episode 1100 | Avg Reward (last 100): -20.26 | Loss: -13.0890
Episode 1200 | Avg Reward (last 100): 2.23 | Loss: -6.7524
Episode 1300 | Avg Reward (last 100): 20.74 | Loss: 1.2829
Episode 1400 | Avg Reward (last 100): 36.60 | Loss: -3.2526
Episode 1500 | Avg Reward (last 100): 47.99 | Loss: -8.6682
Episode 1600 | Avg Reward (last 100): 